# Bias Detection – Training on Google Colab

## Prerequisites
Before running this notebook, EC2 must have already run preprocessing:
```bash
PREPROCESS_ONLY=true LOCAL=false BUCKET=your-bucket bash run_pipeline.sh
```
This uploads `train.csv`, `val.csv`, `test.csv`, and `preprocessed_data/label_encoder.pkl` to S3.

## Flow
1. Pull preprocessed splits from S3
2. Train BERT or DistilBERT
3. Push model artifact to S3

## After This Notebook
SSH into EC2 and run:
```bash
SKIP_TRAIN=true LOCAL=false BUCKET=your-bucket MODEL=bert bash run_pipeline.sh
```

## Configuration

In [ ]:
BUCKET = "your-s3-bucket-name"  # <-- change this
MODEL  = "distilbert"            # "bert" or "distilbert"

# Training hyperparameters
MAX_LEN = 256
BATCH   = 16
EPOCHS  = 3
LR      = 2e-5

## Install Dependencies

In [ ]:
!pip install -q transformers scikit-learn joblib clearml boto3

## AWS Credentials

In Colab: go to the **Secrets** panel (🔑 icon in the left sidebar) and add:
- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_DEFAULT_REGION`

In [ ]:
import os
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = userdata.get("AWS_DEFAULT_REGION")

## Imports

In [ ]:
import io
import os
import joblib
import boto3
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from clearml import Task

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Pull Preprocessed Data from S3
Pulls the splits and label encoder that EC2 generated via `preprocess.py`.

In [ ]:
s3 = boto3.client("s3")

def pull_csv(key):
    print(f"Pulling s3://{BUCKET}/{key} ...")
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_csv(obj["Body"])

train_df = pull_csv("train.csv")
print(f"Train: {train_df.shape}")

print(f"Pulling s3://{BUCKET}/preprocessed_data/label_encoder.pkl ...")
le_obj = s3.get_object(Bucket=BUCKET, Key="preprocessed_data/label_encoder.pkl")
le = joblib.load(io.BytesIO(le_obj["Body"].read()))
print(f"Labels: {list(le.classes_)}")

## Dataset & DataLoader

In [ ]:
MODEL_NAME = "bert-base-uncased" if MODEL == "bert" else "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts, self.labels, self.tokenizer = texts, labels, tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

X_train = train_df["page_text"].values
y_train = train_df["bias"].values

train_loader = DataLoader(TextDataset(X_train, y_train, tokenizer), batch_size=BATCH, shuffle=True)
print(f"Training batches: {len(train_loader)}")

## Train

In [ ]:
task = Task.init(
    project_name="Bias Detection",
    task_name=f"train_{MODEL}_colab",
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le.classes_),
).to(device)

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device),
        )
        outputs.loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")
    task.get_logger().report_scalar("train/loss", "loss", avg_loss, epoch)

## Save Model & Upload to S3

In [ ]:
save_path = f"saved_models/{MODEL}"
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved locally to {save_path}")

def upload_dir_to_s3(local_dir, s3_prefix):
    for root, _, files in os.walk(local_dir):
        for file in files:
            local_path = os.path.join(root, file)
            s3_key = os.path.join(s3_prefix, os.path.relpath(local_path, local_dir))
            print(f"  Uploading {local_path} -> s3://{BUCKET}/{s3_key}")
            s3.upload_file(local_path, BUCKET, s3_key)

print("\nUploading model to S3...")
upload_dir_to_s3(save_path, f"saved_models/{MODEL}")

print("\nDone. Now SSH into EC2 and run:")
print(f"  SKIP_TRAIN=true LOCAL=false BUCKET={BUCKET} MODEL={MODEL} bash run_pipeline.sh")